# LF-CBM soft recall section (concepts instead of attributes)

This section mirrors the outputs of the attribute-based notebook, but replaces:

- **Attribute name** → **LF-CBM concept string**
- **Ground-truth y ∈ {0,1}** → **soft pseudo ground-truth weight**  \tilde P_{i,j} derived from CLIP similarity
- **Recall on positives** → **soft recall**: weighted average of predicted concept probability on pseudo-positive evidence

We still keep a binary notion of “positives” for matched-pair sampling:
- y_bin = 1[ \tilde P > 0 ]
so we can reuse the same matched-pair evaluation structure as the old notebook.

Outputs we will reproduce (old → new equivalents):

Old attribute notebook outputs:
1) per-attribute `info_df`  → per-concept `lf_info_df`
2) per-(attr,pair) `pairs_df` → per-(concept,pair) `lf_pairs_df`
3) per-(attr,species) `species_df` → per-(concept,species) `lf_species_df`
4) `summary` over attrs → `lf_summary` over concepts
5) `top_pairs(...)` tables → `top_pairs_lf(...)` tables

Note: there is no “test_acc” in the same sense, because we are not predicting human labels; we will report:
- **mean_soft_recall** (overall)
- **mean_gap** across matched species pairs (same meaning as before)
- (optional) **corr(pred, Ptilde)** as a sanity check.

For each attribute:

1. Train a linear probe on top of frozen visual features.
2. Evaluate the probe on held-out test images.
3. Group test images by species.
4. For pairs of species:
   - Subsample images so that both species have the same number of
     attribute-positive and attribute-negative examples.
   - Compute recall on attribute-positive images for each species.
5. Measure the recall gap between species.

## FIltering out Concepts formed before Species

In [ ]:
import json
import random
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim


ROOT = Path("/scratch/network/cr7998/cv_emergence_project")
CUB  = ROOT / "data" / "CUB_200_2011"

FEAT_FINE = ROOT / "features" / "resnet50_cub_fine"     # or resnet50_cub
assert FEAT_FINE.exists()

ATTR_TXT = ROOT / "data" / "attributes.txt"   # attr_id -> attr_name like has_primary_color::yellow

BASE_FEAT = ROOT / "features" / "resnet50_cub_fine"
CBM_FEAT  = ROOT / "features" / "resnet50_cub_cbm_fine"

assert CUB.exists(), f"Missing CUB folder: {CUB}"
assert ATTR_TXT.exists(), f"Missing attributes.txt: {ATTR_TXT}"
assert BASE_FEAT.exists(), f"Missing baseline fine features: {BASE_FEAT}"
assert CBM_FEAT.exists(), f"Missing cbm fine features: {CBM_FEAT}"

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
def align_lf_to_feature_order(lf_image_ids: np.ndarray, lf_array: np.ndarray, feature_image_ids: list[int]):
    """
    Align lf_array (in LF ImageFolder order) to feature order (labels_*.pt order).
    Drops any feature_image_ids not present in lf_image_ids.

    Returns:
      kept_feature_image_ids: list[int] (feature order but only those present in LF)
      aligned_lf_array: np.ndarray (same first-dim length as kept_feature_image_ids)
      keep_feat_idx: np.ndarray indices into feature arrays to keep
    """
    lf_pos = {int(img_id): i for i, img_id in enumerate(lf_image_ids)}
    keep_feat_idx = []
    take_lf_idx = []

    for k, img_id in enumerate(feature_image_ids):
        j = lf_pos.get(int(img_id), None)
        if j is None:
            continue
        keep_feat_idx.append(k)
        take_lf_idx.append(j)

    keep_feat_idx = np.asarray(keep_feat_idx, dtype=int)
    take_lf_idx   = np.asarray(take_lf_idx, dtype=int)

    kept_feature_image_ids = [int(feature_image_ids[k]) for k in keep_feat_idx]
    aligned_lf_array = lf_array[take_lf_idx]
    return kept_feature_image_ids, aligned_lf_array, keep_feat_idx

In [ ]:
LAYERS = [
    "conv1",
    "layer1.0","layer1.1","layer1.2",
    "layer2.0","layer2.1","layer2.2","layer2.3",
    "layer3.0","layer3.1","layer3.2","layer3.3","layer3.4","layer3.5",
    "layer4.0","layer4.1","layer4.2",
    "avgpool",
]

def load_feat(layer: str, split: str):
    p = FEAT_DIR / f"{layer}_{split}.pt"
    X = torch.load(p, map_location="cpu", weights_only=False)
    if not torch.is_tensor(X):
        X = torch.as_tensor(X)
    return X.float()

# preload features once (full ordering as saved)
feats_tr = {layer: load_feat(layer, "train") for layer in LAYERS}
feats_te = {layer: load_feat(layer, "test")  for layer in LAYERS}

In [ ]:
obj_te = torch.load(FEAT_DIR / "labels_test.pt",  map_location="cpu", weights_only=False)
obj_tr = torch.load(FEAT_DIR / "labels_train.pt", map_location="cpu", weights_only=False)

test_image_ids  = obj_te["image_ids"].detach().cpu().numpy().astype(int).tolist()
train_image_ids = obj_tr["image_ids"].detach().cpu().numpy().astype(int).tolist()

ysp_te_file = obj_te["labels"].detach().cpu().numpy().astype(int).reshape(-1)
ysp_tr_file = obj_tr["labels"].detach().cpu().numpy().astype(int).reshape(-1)

print("train/test sizes:", len(train_image_ids), len(test_image_ids))

# sanity against feature row counts
assert feats_tr[LAYERS[0]].shape[0] == len(train_image_ids)
assert feats_te[LAYERS[0]].shape[0] == len(test_image_ids)
print("Feature row counts match image_id lists")

# sanity with meta: is file 0-based or 1-based?
meta_idx = meta.set_index("image_id")
y_meta_1based = np.array([int(meta_idx.loc[int(i), "species_id"]) for i in test_image_ids], dtype=int)
m0 = float(np.mean(ysp_te_file == (y_meta_1based - 1)))
m1 = float(np.mean(ysp_te_file == (y_meta_1based)))
print("labels_test match meta (0-based):", m0, " (1-based):", m1)

# choose canonical (almost surely 0-based in your saved labels)
ysp_tr = ysp_tr_file
ysp_te = ysp_te_file

In [ ]:
# Build CLIP pseudo-labels for BOTH splits in LF order (train quantile threshold)
# (Requires: final_cols, lf_ids_train, lf_ids_test, clip_save_name_tr/te, text_save_name already defined earlier)
img_tr = torch.load(LFCBM_REPO/clip_save_name_tr, map_location="cpu", weights_only=False).float()
img_te = torch.load(LFCBM_REPO/clip_save_name_te, map_location="cpu", weights_only=False).float()
txt_all = torch.load(LFCBM_REPO/text_save_name,  map_location="cpu", weights_only=False).float()

img_tr = img_tr / img_tr.norm(dim=1, keepdim=True)
img_te = img_te / img_te.norm(dim=1, keepdim=True)
txt_all = txt_all / txt_all.norm(dim=1, keepdim=True)

P_full_tr = img_tr @ txt_all.T
P_full_te = img_te @ txt_all.T

P_tr = P_full_tr[:, final_cols]
P_te = P_full_te[:, final_cols]

def make_Ptilde(P):
    P_pos = torch.clamp(P, min=0.0)
    den = torch.clamp(P_pos.max(dim=0, keepdim=True).values, min=1e-8)
    return (P_pos / den).cpu().numpy()

Ptilde_tr_lf = make_Ptilde(P_tr)
Ptilde_te_lf = make_Ptilde(P_te)

q = 0.90
taus_tr = np.quantile(Ptilde_tr_lf, q, axis=0)
ybin_tr_lf = (Ptilde_tr_lf > taus_tr[None, :]).astype(np.int32)
ybin_te_lf = (Ptilde_te_lf > taus_tr[None, :]).astype(np.int32)

print("Pseudo-label pos rate train (min/med/max):",
      float(ybin_tr_lf.mean(axis=0).min()),
      float(np.median(ybin_tr_lf.mean(axis=0))),
      float(ybin_tr_lf.mean(axis=0).max()))

In [ ]:
# Align LF concept weights / labels to FEATURE order (dropping grayscale-missing ids)
train_ids_keep, Ptilde_tr, keep_tr_idx = align_lf_to_feature_order(lf_ids_train, Ptilde_tr_lf, train_image_ids)
test_ids_keep,  Ptilde_te, keep_te_idx = align_lf_to_feature_order(lf_ids_test,  Ptilde_te_lf,  test_image_ids)

_, ybin_tr, _ = align_lf_to_feature_order(lf_ids_train, ybin_tr_lf, train_image_ids)
_, ybin_te, _ = align_lf_to_feature_order(lf_ids_test,  ybin_te_lf,  test_image_ids)

print("kept train:", len(train_ids_keep), "dropped:", len(train_image_ids) - len(train_ids_keep))
print("kept test :", len(test_ids_keep),  "dropped:", len(test_image_ids)  - len(test_ids_keep))

# subset feature-side ordering to match LF-present subset
train_image_ids = train_ids_keep
test_image_ids  = test_ids_keep

# subset species labels + features to match
ysp_tr = np.asarray(ysp_tr)[keep_tr_idx]
ysp_te = np.asarray(ysp_te)[keep_te_idx]

feats_tr = {L: feats_tr[L][keep_tr_idx] for L in feats_tr}
feats_te = {L: feats_te[L][keep_te_idx] for L in feats_te}

# final sanity
Ntr = len(train_image_ids)
Nte = len(test_image_ids)
assert feats_tr[LAYERS[0]].shape[0] == Ntr
assert feats_te[LAYERS[0]].shape[0] == Nte
assert Ptilde_tr.shape[0] == ybin_tr.shape[0] == Ntr == len(ysp_tr)
assert Ptilde_te.shape[0] == ybin_te.shape[0] == Nte == len(ysp_te)
print("alignment OK")
print("Aligned:", Ptilde_tr.shape, Ptilde_te.shape, ybin_tr.shape, ybin_te.shape)

In [ ]:
# ----- Species emergence (USE feats_tr/feats_te already filtered & aligned) -----

# Build species labels directly from the filtered ids (train_image_ids/test_image_ids are already filtered)
meta_idx = meta.set_index("image_id")
ysp_tr = np.array([int(meta_idx.loc[int(i), "species_id"]) - 1 for i in train_image_ids], dtype=np.int64)
ysp_te = np.array([int(meta_idx.loc[int(i), "species_id"]) - 1 for i in test_image_ids],  dtype=np.int64)

print("species labels:", ysp_tr.shape, ysp_te.shape)

def train_linear_probe_multiclass(Xtr, ytr, Xte, yte, *, epochs=6, lr=3e-3, wd=1e-4, seed=0, device=None):
    torch.manual_seed(seed)
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    Xtr_t = torch.as_tensor(Xtr, dtype=torch.float32, device=device)
    ytr_t = torch.as_tensor(ytr, dtype=torch.long, device=device)
    Xte_t = torch.as_tensor(Xte, dtype=torch.float32, device=device)

    d = Xtr_t.shape[1]
    C = int(ytr_t.max().item()) + 1

    model = nn.Linear(d, C).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xtr_t), ytr_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(Xte_t).argmax(dim=1).detach().cpu().numpy()

    return float((pred == yte).mean())

def sharp_rise_idx(vals: np.ndarray) -> int:
    v = pd.Series(np.asarray(vals, dtype=float)).ffill().bfill().to_numpy()
    diffs = np.diff(v)
    return int(np.argmax(diffs)) + 1

# Species curve across layers (IMPORTANT: use feats_tr/feats_te)
species_curve = []
for layer in LAYERS:
    acc = train_linear_probe_multiclass(feats_tr[layer], ysp_tr, feats_te[layer], ysp_te, epochs=6, seed=0)
    species_curve.append(acc)

species_curve = np.asarray(species_curve, dtype=float)
species_emerge_idx = sharp_rise_idx(species_curve)

print("Species emergence (sharp-jump):", species_emerge_idx, "->", LAYERS[species_emerge_idx])
for l, a in zip(LAYERS, species_curve):
    print(f"{l:10s} {a:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# ---------- metrics ----------
def balanced_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(int).reshape(-1)
    y_pred = y_pred.astype(int).reshape(-1)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return 0.5 * (tpr + tnr)

def sharp_rise_idx(vals: np.ndarray) -> int:
    v = pd.Series(np.asarray(vals, dtype=float)).ffill().bfill().to_numpy()
    diffs = np.diff(v)
    return int(np.argmax(diffs)) + 1

def frac_of_final_idx(vals: np.ndarray, frac: float = 0.90) -> int:
    """
    earliest index i such that vals[i] >= frac * vals[-1]
    if never reaches, return last index.
    """
    v = pd.Series(np.asarray(vals, dtype=float)).ffill().bfill().to_numpy()
    target = frac * float(v[-1])
    for i, x in enumerate(v):
        if float(x) >= target:
            return int(i)
    return int(len(v) - 1)

# ---------- weighted probe ----------
def train_linear_probe_binary_weighted(
    Xtr, ytr, Xte, yte, *,
    epochs=8, lr=3e-3, wd=1e-4, seed=0,
    threshold=0.5,
):
    """
    Binary linear probe with pos_weight so it doesn't collapse on imbalanced labels.
    Returns: (balanced_acc, plain_acc, pred_pos_rate_test, probs_test)
    """
    torch.manual_seed(seed)

    Xtr_t = torch.as_tensor(Xtr, dtype=torch.float32)
    Xte_t = torch.as_tensor(Xte, dtype=torch.float32)

    ytr_np = np.asarray(ytr, dtype=np.int32).reshape(-1)
    yte_np = np.asarray(yte, dtype=np.int32).reshape(-1)

    ytr_t = torch.as_tensor(ytr_np, dtype=torch.float32).view(-1, 1)

    d = int(Xtr_t.shape[1])
    model = nn.Linear(d, 1)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    # pos_weight = (#neg / #pos)
    pos = float(ytr_np.sum())
    neg = float(len(ytr_np) - ytr_np.sum())
    if pos <= 0:
        # degenerate; can't train
        probs = np.zeros_like(yte_np, dtype=float)
        pred = np.zeros_like(yte_np, dtype=int)
        return 0.5, float((pred == yte_np).mean()), 0.0, probs

    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(Xtr_t)
        loss = loss_fn(logits, ytr_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).view(-1).cpu().numpy()

    pred = (probs >= threshold).astype(int)
    ba = balanced_accuracy(yte_np, pred)
    acc = float((pred == yte_np).mean())
    pred_pos_rate = float(pred.mean())
    return float(ba), float(acc), float(pred_pos_rate), probs

# ---------- concept emergence ----------
def concept_emergence(
    concept_names,
    ybin_tr, ybin_te,
    *,
    max_concepts=None,
    min_pos_rate=0.03, max_pos_rate=0.97,
    epochs=8,
    seed=0,
    frac=0.90,
):
    """
    Computes per-concept curves over layers using BALANCED ACC,
    and returns two emergence indices:
      - sharp-jump emergence (argmax delta)
      - frac-of-final emergence (>= frac * final)
    """
    rows = []
    M = ybin_tr.shape[1]
    if max_concepts is not None:
        M = min(M, int(max_concepts))

    for j in range(M):
        ytr = np.asarray(ybin_tr[:, j], dtype=np.int32)
        yte = np.asarray(ybin_te[:, j], dtype=np.int32)

        rate_tr = float(ytr.mean())
        rate_te = float(yte.mean())
        if not (min_pos_rate <= rate_tr <= max_pos_rate):
            continue

        ba_curve = []
        acc_curve = []
        ppos_curve = []

        for layer in LAYERS:
            ba, acc, ppos, _ = train_linear_probe_binary_weighted(
                feats_tr[layer], ytr,
                feats_te[layer], yte,
                epochs=epochs,
                seed=seed,
            )
            ba_curve.append(ba)
            acc_curve.append(acc)
            ppos_curve.append(ppos)

        ba_curve = np.asarray(ba_curve, dtype=float)
        diffs = np.diff(ba_curve)

        e_jump = sharp_rise_idx(ba_curve)
        e_frac = frac_of_final_idx(ba_curve, frac=frac)

        rows.append({
            "j": int(j),
            "concept": str(concept_names[j]) if j < len(concept_names) else f"concept_{j}",
            "pos_rate_tr": rate_tr,
            "pos_rate_te": rate_te,
            "emerge_idx_jump": int(e_jump),
            "emerge_layer_jump": LAYERS[int(e_jump)],
            "emerge_idx_frac": int(e_frac),
            "emerge_layer_frac": LAYERS[int(e_frac)],
            "final_ba": float(ba_curve[-1]),
            "max_jump": float(diffs.max()) if len(diffs) else 0.0,
        })

        if (len(rows) % 25) == 0:
            print("kept", len(rows), "concepts so far")

    return pd.DataFrame(rows)

# Run emergence
lf_emerge_df = concept_emergence(
    lf_concepts_final,
    ybin_tr, ybin_te,
    max_concepts=400,
    min_pos_rate=0.03, max_pos_rate=0.97,
    epochs=8,
    seed=0,
    frac=0.90,
)

print("concepts kept (rate-filtered):", len(lf_emerge_df))

# Species emergence index from your species_curve
print("Species emerge idx:", species_emerge_idx, "layer:", LAYERS[species_emerge_idx])

# Post-species using sharp-jump
POST_jump = lf_emerge_df[lf_emerge_df["emerge_idx_jump"] > species_emerge_idx].copy()
POST_jump = POST_jump.sort_values(["emerge_idx_jump","final_ba"], ascending=[True, False]).reset_index(drop=True)
print("Post-species concepts (sharp-jump):", len(POST_jump))
print("First 15 (concept, j, emerge_layer_jump):", list(zip(
    POST_jump["concept"].head(15).tolist(),
    POST_jump["j"].head(15).tolist(),
    POST_jump["emerge_layer_jump"].head(15).tolist()
)))

# Post-species using 0.9-of-final
POST_frac = lf_emerge_df[lf_emerge_df["emerge_idx_frac"] > species_emerge_idx].copy()
POST_frac = POST_frac.sort_values(["emerge_idx_frac","final_ba"], ascending=[True, False]).reset_index(drop=True)
print("Post-species concepts (0.9-of-final):", len(POST_frac))
print("First 15 (concept, j, emerge_layer_frac):", list(zip(
    POST_frac["concept"].head(15).tolist(),
    POST_frac["j"].head(15).tolist(),
    POST_frac["emerge_layer_frac"].head(15).tolist()
)))

# Show full emergence table (sorted)
lf_emerge_sorted = lf_emerge_df.sort_values(["emerge_idx_frac","final_ba"], ascending=[True, False]).reset_index(drop=True)
print("\nTop 30 concepts by 0.9-of-final emergence:")
print(lf_emerge_sorted[["j","concept","emerge_idx_frac","emerge_layer_frac","final_ba","max_jump","pos_rate_te"]].head(30))

In [ ]:
print("Species emerge idx (sharp-jump):", species_emerge_idx_jump, "->", LAYERS[species_emerge_idx_jump])

post_jump_all = lf_df[lf_df["jump_all_idx"] >= species_emerge_idx_jump]
post_jump_noconv1 = lf_df[lf_df["jump_no_conv1_idx"] >= species_emerge_idx_jump]
post_frac90 = lf_df[lf_df["frac90_idx"] >= species_emerge_idx_jump]

print("Post-species counts:")
print("  jump_all      :", len(post_jump_all))
print("  jump_no_conv1 :", len(post_jump_noconv1))
print("  frac90        :", len(post_frac90))

In [ ]:
import matplotlib.pyplot as plt

def plot_concept_curve(j: int, title_extra=""):
    ytr = np.asarray(ybin_tr[:, j], dtype=np.int32)
    yte = np.asarray(ybin_te[:, j], dtype=np.int32)

    ba_curve, acc_curve, ppos_curve = [], [], []
    for layer in LAYERS:
        ba, acc, ppos, _ = train_linear_probe_binary_weighted(
            feats_tr[layer], ytr, feats_te[layer], yte,
            epochs=8, seed=0
        )
        ba_curve.append(ba); acc_curve.append(acc); ppos_curve.append(ppos)

    ba_curve = np.asarray(ba_curve, float)
    e_jump = sharp_rise_idx(ba_curve)
    e_frac = frac_of_final_idx(ba_curve, frac=0.90)

    plt.figure()
    plt.plot(ba_curve, marker="o", label="Balanced accuracy (BA)")
    plt.plot(acc_curve, marker="o", label="Plain accuracy")
    plt.plot(ppos_curve, marker="o", label="Predicted positive rate (test)")
    plt.xticks(range(len(LAYERS)), LAYERS, rotation=45, ha="right")
    plt.ylim(0, 1.0)
    name = lf_concepts_final[j] if j < len(lf_concepts_final) else f"concept_{j}"
    plt.title(f"LF concept j={j}: {name}\njump={LAYERS[e_jump]}, frac90={LAYERS[e_frac]} {title_extra}")
    plt.legend()
    plt.tight_layout()
    plt.show()

# examples
plot_concept_curve(0)
plot_concept_curve(241)

In [ ]:
def debug_one_concept(j=0, layer="avgpool", epochs=20):
    ytr = ybin_tr[:, j].astype(int)
    yte = ybin_te[:, j].astype(int)

    print("concept j:", j, "| name:", lf_concepts_final[j])
    print("pos rate train:", ytr.mean(), "test:", yte.mean())

    # constant baselines
    ba_all0 = balanced_accuracy(yte, np.zeros_like(yte))
    ba_all1 = balanced_accuracy(yte, np.ones_like(yte))
    print("BA all-0:", ba_all0, "BA all-1:", ba_all1)

    Xtr = feats_tr[layer]
    Xte = feats_te[layer]

    torch.manual_seed(0)
    model = nn.Linear(Xtr.shape[1], 1).to(device)
    opt = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    Xtr_t = Xtr.to(device)
    ytr_t = torch.as_tensor(ytr, dtype=torch.float32, device=device).view(-1, 1)
    Xte_t = Xte.to(device)

    losses = []
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(Xtr_t)
        loss = loss_fn(logits, ytr_t)
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))

    model.eval()
    with torch.no_grad():
        log_te = model(Xte_t).view(-1).detach().cpu().numpy()
    pred = (log_te >= 0.0).astype(int)

    print("loss first/last:", losses[0], losses[-1])
    print("pred pos rate (test):", pred.mean())
    print("balanced acc:", balanced_accuracy(yte, pred))

debug_one_concept(j=0, layer="avgpool", epochs=30)

### Original result:
OK: filtered feature rows match labels: 5990 5790

Species curve (test acc per layer):

| layer    | test_acc |
|----------|----------|
| conv1    | 0.0067 |
| layer1.0 | 0.0176 |
| layer1.1 | 0.0150 |
| layer1.2 | 0.0152 |
| layer2.0 | 0.0244 |
| layer2.1 | 0.0306 |
| layer2.2 | 0.0263 |
| layer2.3 | 0.0302 |
| layer3.0 | 0.0539 |
| layer3.1 | 0.0660 |
| layer3.2 | 0.0684 |
| layer3.3 | 0.0679 |
| layer3.4 | 0.0667 |
| layer3.5 | 0.0746 |
| layer4.0 | 0.2832 |
| layer4.1 | 0.3667 |
| layer4.2 | 0.6207 |
| avgpool  | 0.6207 |

Species sharp-jump emergence: 16 -> layer4.2 | max jump = 0.2541

Concepts total: 251  
Concepts skipped (degenerate y): 0  
Concepts with curves: 251

Counts relative to species (sharp-jump):
- before species: 251
- at species    : 0
- after species : 0

Top 30 concepts (sorted by emerge_idx, then final_acc):

| j | concept | emerge_idx | emerge_layer | final_acc | max_jump | pos_rate_te |
|---|--------|------------|--------------|-----------|----------|-------------|
| 250 | yellowish-brown wings | 1 | layer1.0 | 0.987910 | 0.0 | 0.012090 |
| 235 | white underwings | 1 | layer1.0 | 0.987565 | 0.0 | 0.012435 |
| 237 | wing bars | 1 | layer1.0 | 0.980829 | 0.0 | 0.019171 |
| 185 | iridescent feathers | 1 | layer1.0 | 0.975993 | 0.0 | 0.024007 |
| 189 | long black tail | 1 | layer1.0 | 0.975820 | 0.0 | 0.024180 |
| 25 | a black or dark grey plumage | 1 | layer1.0 | 0.972193 | 0.0 | 0.027807 |
| 27 | a black tail | 1 | layer1.0 | 0.971675 | 0.0 | 0.028325 |
| 62 | a large, hawk-like body | 1 | layer1.0 | 0.968048 | 0.0 | 0.031952 |
| 180 | greenish upperparts | 1 | layer1.0 | 0.965630 | 0.0 | 0.034370 |
| 63 | a large, orange bill | 1 | layer1.0 | 0.960276 | 0.0 | 0.039724 |
| 236 | white wingbars | 1 | layer1.0 | 0.959931 | 0.0 | 0.040069 |
| 156 | black wings with white spots | 1 | layer1.0 | 0.959585 | 0.0 | 0.040415 |
| 177 | green head | 1 | layer1.0 | 0.954404 | 0.0 | 0.045596 |
| 22 | a black face | 1 | layer1.0 | 0.953022 | 0.0 | 0.046978 |
| 173 | gray upperparts | 1 | layer1.0 | 0.950950 | 0.0 | 0.049050 |
| 30 | a blue head | 1 | layer1.0 | 0.949914 | 0.0 | 0.050086 |
| 17 | a black breast | 1 | layer1.0 | 0.948705 | 0.0 | 0.051295 |
| 99 | a small, blue body | 1 | layer1.0 | 0.948705 | 0.0 | 0.051295 |
| 20 | a black collar | 1 | layer1.0 | 0.947668 | 0.0 | 0.052332 |
| 91 | a rusty-brown breast band | 1 | layer1.0 | 0.946114 | 0.0 | 0.053886 |
| 3 | All-dark plumage | 1 | layer1.0 | 0.945596 | 0.0 | 0.054404 |
| 159 | blue upperparts | 1 | layer1.0 | 0.945423 | 0.0 | 0.054577 |
| 138 | a yellow throat and breast | 1 | layer1.0 | 0.943869 | 0.0 | 0.056131 |
| 216 | shaggy feathers | 1 | layer1.0 | 0.943869 | 0.0 | 0.056131 |
| 149 | black feathers | 1 | layer1.0 | 0.941796 | 0.0 | 0.058204 |
| 244 | yellow wings with black bars | 1 | layer1.0 | 0.941623 | 0.0 | 0.058377 |
| 167 | dark gray or black upperparts | 1 | layer1.0 | 0.940933 | 0.0 | 0.059067 |
| 13 | a black back | 1 | layer1.0 | 0.940587 | 0.0 | 0.059413 |
| 192 | long, black ear tufts | 1 | layer1.0 | 0.940587 | 0.0 | 0.059413 |
| 164 | brownish wings with white bars | 1 | layer1.0 | 0.938860 | 0.0 | 0.061140 |

ALL concept emergences:
251 rows × 7 columns

### Why the LF concept emergence initially looked wrong 

---

## What we were trying to measure

For both **species** and **concepts**, we want to answer:

> *At which backbone layer does a linear probe first become good at predicting this label?*

We operationalize this by:
1. Training a linear probe on features from each layer.
2. Measuring probe performance on a held-out split.
3. Defining an **emergence layer** from the layerwise performance curve.

This worked as expected for **species** and for **human-annotated attributes**, but initially failed for **LF concepts**.

---

## What went wrong for LF concepts

### 1. Extreme class imbalance in LF pseudo-labels

LF concept labels are derived from CLIP scores (e.g., “top-quantile = present”).  
In practice, many concepts ended up with very low prevalence:

- Some concepts had <2% positive examples.
- Others had very high or very low base rates.

This matters because the probe was trained with **binary cross-entropy** and evaluated with **plain accuracy**.

---

### 2. What the probe actually optimized

For a binary probe with cross-entropy loss:

$$
\mathcal{L} = -\mathbb{E}\left[y \log \sigma(z) + (1-y)\log(1-\sigma(z))\right]
$$

If positives are rare, the loss is minimized by predicting the **base rate**:

$$
\sigma(z^*) = \Pr(y=1)
$$

When $\Pr(y=1)$ is very small, thresholding at 0.5 yields:
- predict “absent” for every example
- accuracy $\approx$ fraction of negatives

So the probe achieved very high accuracy **without using features at all**.

---

### 3. Consequence for emergence detection

Because the probe learned the same trivial solution at every layer:

- Accuracy curves were almost flat.
- The “largest jump” heuristic (`sharp_rise_idx`) always returned the first layer.
- All concepts appeared to “emerge” immediately.
- Result: **0 concepts emerged after species**, but for the wrong reason.

This was not a property of the representation — it was a metric artifact.

---

## Why this did NOT happen for species or standard CBM attributes

### Species (multiclass case)

Species probes use **multiclass cross-entropy** with ~200 roughly balanced classes.

- A constant predictor achieves ~0.5% accuracy.
- The probe *must* use features to reduce loss.
- Layerwise accuracy curves are meaningful.

### Human-annotated attributes (standard CBM)

Attributes:
- Have real human labels.
- Are filtered by certainty.
- Have reasonable prevalence.

So the trivial “always negative” solution is not competitive, and probes actually learn.

---

## What we changed

### 1. Fixed label construction

For LF concepts, pseudo-labels are now constructed **per-concept using quantiles** (e.g., top 10%), ensuring comparable prevalence across concepts.

This prevents accidental 1–2% positive rates.

---

### 2. Changed the evaluation metric

Instead of plain accuracy, we use a metric robust to imbalance, such as:

- **Balanced accuracy**  

$$
\frac{1}{2}(\text{TPR} + \text{TNR})
$$

- or **AUC**

This prevents the trivial constant predictor from scoring highly.

---

### 3. Adjusted emergence definition

The “largest jump” heuristic is unstable when curves are smooth or flat.

We instead define emergence as:

> The earliest layer where performance reaches a fixed fraction (e.g., 90%) of final performance.

This produces stable and interpretable emergence layers.

---

## Summary comparison

| Setting | Label source | Class balance | Metric | Emergence meaningful? |
|----------|-------------|---------------|--------|-----------------------|
| Species | Ground truth | Balanced | Accuracy | Yes |
| CBM attributes | Human-annotated | Reasonable | Accuracy | Yes |
| LF concepts (before) | CLIP pseudo-labels | Highly imbalanced | Accuracy | No |
| LF concepts (after) | Quantile-balanced | Controlled | Balanced acc / AUC | Yes |

---

## Key takeaway

The initial “no post-species concepts” result was **not a representation finding**.  
It was caused by combining:

- extremely imbalanced pseudo-labels
- cross-entropy training
- plain accuracy evaluation

Once labels and metrics are aligned with the question being asked, LF concept emergence behaves sensibly and can be compared meaningfully to species and CBM attributes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

def smooth_1d(v, win=3):
    """Simple centered moving average; win=1 means no smoothing."""
    v = np.asarray(v, dtype=float)
    if win <= 1:
        return v
    s = pd.Series(v).rolling(win, center=True, min_periods=1).mean().to_numpy()
    return s

def sharp_jump_idx(vals, *, smooth_win=3, min_abs_gain=0.0):
    """
    Sharp-jump emergence:
      - optionally smooth curve
      - take first-difference
      - choose argmax jump
      - optional guard: require jump >= min_abs_gain else return 0
    """
    v = smooth_1d(vals, win=smooth_win)
    diffs = np.diff(v)
    j = int(np.argmax(diffs)) + 1  # +1 because diff is between layers
    if diffs[j-1] < min_abs_gain:
        return 0
    return j

## Finding Recall Gap

In [ ]:
LFCBM_REPO = Path("/scratch/network/cr7998/Label-free-CBM/")         
LFCBM_RUN_DIR = LFCBM_REPO / "saved_models" / "cub_cbm_2026_02_06_11_37" 

assert LFCBM_REPO.exists(), f"Missing: {LFCBM_REPO}"
assert LFCBM_RUN_DIR.exists(), f"Missing: {LFCBM_RUN_DIR}"

# Allow importing lfcbm utils/data_utils/cbm if needed
if str(LFCBM_REPO) not in sys.path:
    sys.path.insert(0, str(LFCBM_REPO))

PM_SUFFIX = {"max": "_max", "avg": ""}

def lf_get_save_names(clip_name, target_name, target_layer, d_probe, concept_set, pool_mode, save_dir):
    """
    Local copy of LF-CBM utils.get_save_names so we don't import LF-CBM repo deps.
    """
    save_dir = str(save_dir)
    if str(target_name).startswith("clip_"):
        target_save_name = f"{save_dir}/{d_probe}_{str(target_name).replace('/', '')}.pt"
    else:
        target_save_name = f"{save_dir}/{d_probe}_{target_name}_{target_layer}{PM_SUFFIX[pool_mode]}.pt"
    clip_save_name = f"{save_dir}/{d_probe}_clip_{str(clip_name).replace('/', '')}.pt"
    concept_set_name = Path(concept_set).name.split(".")[0]
    text_save_name = f"{save_dir}/{concept_set_name}_{str(clip_name).replace('/', '')}.pt"
    return target_save_name, clip_save_name, text_save_name

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
def load_species_maps(cub_root: Path):
    classes = pd.read_csv(
        cub_root / "classes.txt",
        sep=r"\s+",
        header=None,
        names=["species_id", "class_name"],
        engine="python"
    )

    def pretty(name: str) -> str:
        return name.split(".", 1)[-1].replace("_", " ")

    return {
        int(r.species_id): pretty(r.class_name)
        for _, r in classes.iterrows()
    }

species_id_to_name = load_species_maps(CUB)

def spname(sid: int) -> str:
    return species_id_to_name.get(int(sid), f"species_{sid}")

In [ ]:
def load_meta(cub_root: Path) -> pd.DataFrame:
    img_species = pd.read_csv(
        cub_root / "image_class_labels.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "species_id"],
        engine="python"
    )

    split_df = pd.read_csv(
        cub_root / "train_test_split.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "is_train"],
        engine="python"
    )

    meta = img_species.merge(split_df, on="image_id")
    meta["species_name"] = meta["species_id"].map(spname)
    return meta

meta = load_meta(CUB)

In [ ]:
def load_image_attr_labels_robust(cub_root: Path) -> pd.DataFrame:
    path = cub_root / "attributes" / "image_attribute_labels.txt"
    rows = []

    with open(path, "r") as f:
        for line in f:
            toks = line.strip().split()
            if len(toks) < 4:
                continue
            rows.append((int(toks[0]), int(toks[1]), int(toks[2]), int(toks[3])))

    return pd.DataFrame(rows, columns=["image_id", "attr_id", "is_present", "certainty"])

img_attr_long = load_image_attr_labels_robust(CUB)

In [ ]:
def load_attr_maps(attr_txt: Path):
    rows = []
    with open(attr_txt, "r") as f:
        for line in f:
            aid, name = line.strip().split(" ", 1)
            rows.append((int(aid), name))
    df = pd.DataFrame(rows, columns=["attr_id", "attr_name"])
    return df, dict(zip(df.attr_name, df.attr_id)), dict(zip(df.attr_id, df.attr_name))

attr_df, attr_name_to_id, attr_id_to_name = load_attr_maps(ATTR_TXT)

## Load LF-CBM run metadata + concept list

We load:
- `args.txt` to know which concept_set and backbone were used.
- `concepts.txt` which is the *final filtered* concept list used by the trained LF-CBM model.

We also load the initial concept set file referenced by args, because the saved CLIP text features correspond to that initial list, and we need to map initial → final indices.

In [ ]:
# Load args from the trained LF-CBM directory
with open(LFCBM_RUN_DIR / "args.txt", "r") as f:
    lf_args = json.load(f)

with open(LFCBM_RUN_DIR / "concepts.txt", "r") as f:
    lf_concepts_final = [x.strip() for x in f.read().split("\n") if x.strip()]

print("Backbone:", lf_args["backbone"])
print("Feature layer:", lf_args["feature_layer"])
print("Clip name:", lf_args["clip_name"])
print("Concept set file:", lf_args["concept_set"])
print("Num final concepts:", len(lf_concepts_final))
print("Example concepts:", lf_concepts_final[:10])

## Build mapping: ImageFolder order → CUB image_id

LF-CBM uses `torchvision.datasets.ImageFolder(data/CUB/test)` ordering.
Your `meta` / attribute labels are keyed by `image_id` from CUB’s official files.

So we must align:
- the **row order** in LF-CBM’s saved activations (ImageFolder order)
with
- the CUB `image_id` values used in your `meta`.

We do this by using CUB’s `images.txt` which maps:
  image_id ↔ relative path like "001.Black_footed_Albatross/Black_Footed_Albatross_0001_796111.jpg"

Then we reconstruct the ImageFolder relative paths and map them to image_id.

In [ ]:
from torchvision import datasets

def load_images_txt_map(cub_root: Path):
    # CUB_200_2011/images.txt
    df = pd.read_csv(
        cub_root / "images.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "relpath"],
        engine="python"
    )
    rel_to_id = dict(zip(df["relpath"].astype(str), df["image_id"].astype(int)))
    return rel_to_id

rel_to_image_id = load_images_txt_map(CUB)

def get_imagefolder_image_ids(imagefolder_root: Path):
    ds = datasets.ImageFolder(imagefolder_root)  # transform irrelevant; we only use .samples
    relpaths = []
    for (abspath, _cls) in ds.samples:
        # Make relpath like CUB uses: "<class_folder>/<filename>"
        abspath = Path(abspath)
        rel = f"{abspath.parent.name}/{abspath.name}"
        relpaths.append(rel)

    image_ids = []
    missing = 0
    for rel in relpaths:
        if rel not in rel_to_image_id:
            missing += 1
            image_ids.append(-1)
        else:
            image_ids.append(rel_to_image_id[rel])
    image_ids = np.array(image_ids, dtype=int)

    assert missing == 0, f"{missing} ImageFolder paths not found in images.txt mapping"
    return image_ids

# LF-CBM datasets.ImageFolder roots (from its data_utils.py):
LFCBM_CUB_TEST = LFCBM_REPO / "data" / "CUB" / "test"
LFCBM_CUB_TRAIN = LFCBM_REPO / "data" / "CUB" / "train"

assert LFCBM_CUB_TEST.exists(), f"Missing: {LFCBM_CUB_TEST}"
assert LFCBM_CUB_TRAIN.exists(), f"Missing: {LFCBM_CUB_TRAIN}"

lf_ids_test = get_imagefolder_image_ids(LFCBM_CUB_TEST)
lf_ids_train = get_imagefolder_image_ids(LFCBM_CUB_TRAIN)

print("LF test ids range:", lf_ids_test.min(), lf_ids_test.max(), "n=", len(lf_ids_test))
print("LF train ids range:", lf_ids_train.min(), lf_ids_train.max(), "n=", len(lf_ids_train))

## Load LF-CBM concept activations `c(x)` (the bottleneck)

Your LF-CBM model stores:
- `W_c.pt`  (projection weights)
- `proj_mean.pt`, `proj_std.pt` (normalization used before final layer)

We will reproduce the exact LF-CBM bottleneck activations used for classification:
  c(x) = (W_c f(x) - mean) / std

Where f(x) are the backbone activations saved in `activation_dir` for the chosen `feature_layer`.

In [ ]:
# Load LF-CBM learned projection and normalization
W_c = torch.load(LFCBM_RUN_DIR / "W_c.pt", map_location="cpu", weights_only=False)
proj_mean = torch.load(LFCBM_RUN_DIR / "proj_mean.pt", map_location="cpu", weights_only=False)
proj_std  = torch.load(LFCBM_RUN_DIR / "proj_std.pt", map_location="cpu", weights_only=False)

W_c = W_c.float()
proj_mean = proj_mean.float()
proj_std = proj_std.float()

print("W_c:", tuple(W_c.shape), "proj_mean:", tuple(proj_mean.shape), "proj_std:", tuple(proj_std.shape))
print("Num concepts implied:", W_c.shape[0], "==", len(lf_concepts_final))

## Load saved backbone activations for train/test and compute c(x)

LF-CBM saved activations with `utils.get_save_names(...)`.
We will use the same helper from LF-CBM repo to locate the `.pt` files, then compute:
- train_c: [N_train, M] bottleneck coords
- test_c : [N_test,  M] bottleneck coords

These are aligned to ImageFolder order, which we mapped to `image_id` arrays (`lf_ids_train`, `lf_ids_test`).

In [ ]:
def resolve_lf_path(p: str | Path) -> Path:
    p = Path(p)
    # if already absolute, keep it
    if p.is_absolute():
        return p
    # otherwise treat as relative to LF-CBM repo root
    return (LFCBM_REPO / p).resolve()

concept_set_path = resolve_lf_path(lf_args["concept_set"])
assert concept_set_path.exists(), f"Missing concept_set: {concept_set_path}"

with open(concept_set_path, "r") as f:
    lf_concepts_initial = [x.strip() for x in f.read().split("\n") if x.strip()]

In [ ]:
# Helper: load initial concept list (the file used to generate saved text features)
concept_set_path = resolve_lf_path(Path(lf_args["concept_set"]))
assert concept_set_path.exists(), f"Missing concept_set: {concept_set_path}"

with open(concept_set_path, "r") as f:
    lf_concepts_initial = [x.strip() for x in f.read().split("\n") if x.strip()]

# Map initial concept index by string
init_index = {c:i for i,c in enumerate(lf_concepts_initial)}

# Build column indices that correspond to final concepts
final_cols = []
missing = []
for c in lf_concepts_final:
    if c not in init_index:
        missing.append(c)
    else:
        final_cols.append(init_index[c])

assert len(missing) == 0, f"Final concepts missing from initial list (should not happen). Examples: {missing[:5]}"
final_cols = np.array(final_cols, dtype=int)

# Locate saved backbone activations (target features) for train and val(test)
activation_dir = lf_args["activation_dir"]
clip_name = lf_args["clip_name"]
backbone = lf_args["backbone"]
feature_layer = lf_args["feature_layer"]
pool_mode = "avg"  # LF-CBM script used avg in train_cbm.py

# d_probe names inside LF-CBM training script:
d_train = lf_args["dataset"] + "_train"  # "cub_train"
d_val   = lf_args["dataset"] + "_val"    # "cub_val"  (this corresponds to ImageFolder test)

# Use lf_get_save_names
target_save_name_tr, clip_save_name_tr, text_save_name = lf_get_save_names(
    clip_name, backbone, feature_layer, d_train, str(concept_set_path), pool_mode, activation_dir
)
target_save_name_te, clip_save_name_te, _ = lf_get_save_names(
    clip_name, backbone, feature_layer, d_val, str(concept_set_path), pool_mode, activation_dir
)

print("Target feats train:", LFCBM_REPO/target_save_name_tr)
print("Target feats test :", LFCBM_REPO/target_save_name_te)

target_tr = torch.load(LFCBM_REPO/target_save_name_tr, map_location="cpu", weights_only=False).float()
target_te = torch.load(LFCBM_REPO/target_save_name_te, map_location="cpu", weights_only=False).float()

# Compute bottleneck activations and normalize exactly like cbm.py
train_c = (target_tr @ W_c.T)  # [N, M]
test_c  = (target_te @ W_c.T)

train_c = (train_c - proj_mean) / proj_std
test_c  = (test_c  - proj_mean) / proj_std

print("train_c:", tuple(train_c.shape), "test_c:", tuple(test_c.shape))

## Compute pseudo-ground-truth concept evidence: Ptilde from CLIP similarity matrix

We compute the LF concept matrix on the test set:
  P = normalize(E_I(x)) · normalize(E_T(t))

Then we select only the final concepts (same ordering as concepts.txt),
and convert to a nonnegative, normalized weight:

  P_pos = max(P, 0)
  Ptilde = P_pos / max(P_pos)    (per concept)   so Ptilde ∈ [0, 1]

This Ptilde is your “soft ground truth evidence” for that concept being present in that image.

In [ ]:
import torch

# Load saved CLIP image features + text features for the concept_set
img_te = torch.load(LFCBM_REPO/clip_save_name_te, map_location="cpu", weights_only=False).float()
txt_all = torch.load(LFCBM_REPO/text_save_name, map_location="cpu", weights_only=False).float()

# Normalize exactly like LF-CBM training code
img_te = img_te / img_te.norm(dim=1, keepdim=True)
txt_all = txt_all / txt_all.norm(dim=1, keepdim=True)

# Full CLIP concept matrix for the initial concept list: P_full [N_test, M_initial]
P_full = img_te @ txt_all.T

# Keep only the final concepts, in the same order as test_c
P = P_full[:, final_cols]  # [N_test, M_final]

# Nonnegative weights for "soft ground truth"
P_pos = torch.clamp(P, min=0.0)

# Scale each concept column to [0,1] (for stable weighting across concepts)
den = torch.clamp(P_pos.max(dim=0, keepdim=True).values, min=1e-8)
Ptilde = (P_pos / den).cpu().numpy()  # [N_test, M_final]

print("Ptilde shape:", Ptilde.shape, "min/max:", float(Ptilde.min()), float(Ptilde.max()))

In [ ]:
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

test_c_np = test_c.detach().cpu().numpy()     # [N_test, M_final]
pred_probs = sigmoid_np(test_c_np)            # [N_test, M_final] scores in (0,1)

assert pred_probs.shape == Ptilde.shape
print("pred_probs:", pred_probs.shape)

In [ ]:
# Binary labels for matched sampling:
# "present" = top (1-q) fraction by CLIP weight for each concept.
q = 0.90  # top 10% positives
taus = np.quantile(Ptilde, q, axis=0)                 # [M_final]
ybin = (Ptilde > taus[None, :]).astype(np.int32)      # [N_test, M_final]

rates = ybin.mean(axis=0)
print("pos rate (min/median/max):", float(rates.min()), float(np.median(rates)), float(rates.max()))

## Define soft recall machinery (species tables + matched pairs), mirroring the old notebook

We now mimic your old functions, but with:

- `pred_prob` = sigmoid(test_c[:, j])     (concept activation → probability-like score)
- `w` = Ptilde[:, j]                      (soft evidence)
- `y_bin` = 1[w > 0]                      (only for matched-pair sampling)

Per-species soft recall:
  soft_recall = sum(pred_prob * w) / sum(w)   over images in that species (and optionally only y_bin==1)

Matched-pairs:
- we match by y_bin counts (positives/negatives) like before,
- and compute soft recall on the y_bin==1 subset using weights w.

In [ ]:
import numpy as np

def build_lf_test_df(meta: pd.DataFrame, image_ids: np.ndarray) -> pd.DataFrame:
    """
    Build a dataframe in EXACTLY the same row order as LF test arrays.
    Requires: meta has one row per image_id with species_id/species_name.
    """
    m = meta.set_index("image_id")[["species_id", "species_name"]]
    rows = []
    for img_id in image_ids:
        sid, sname = m.loc[int(img_id)]
        rows.append((int(img_id), int(sid), str(sname)))
    return pd.DataFrame(rows, columns=["image_id", "species_id", "species_name"])

lf_test_df = build_lf_test_df(meta, lf_ids_test)
lf_test_df.head()

## Per-concept per-species table (new equivalent of `baseline_species` / `cbm_species`)

Old: `species_recall_prevalence_table(df_te, probs, thr=0.5)`
New: `species_soft_recall_table(lf_test_df, pred_prob_j, w_j)`

We’ll output columns analogous to old:
- n, n_pos_bin, prevalence_bin
- soft_recall
- (optional) mean_pred, mean_w

In [ ]:
"""
def species_soft_recall_table(df_test: pd.DataFrame,
                              pred: np.ndarray,
                              w: np.ndarray,
                              ybin: np.ndarray) -> pd.DataFrame:
    ""
    Per-species summary for ONE concept.

    df_test must contain: species_id, species_name  (one row per image, aligned with pred/w/ybin)

    pred:  [N] predicted concept probability in [0,1]
    w:     [N] soft weight (>=0), e.g. Ptilde[:, j]
    ybin:  [N] binary pseudo-label in {0,1}, e.g. (P[:, j] > tau_j)

    soft_recall_s(j) = sum_{i in species s, ybin_i=1} pred_i * w_i  /  sum_{i in species s, ybin_i=1} w_i
    ""
    tmp = df_test[["species_id", "species_name"]].copy()
    tmp["pred"] = pred
    tmp["w"] = w
    tmp["ybin"] = ybin.astype(int)

    def agg_one(g):
        n = len(g)
        n_pos_bin = int(g["ybin"].sum())
        n_neg_bin = int(n - n_pos_bin)
        prevalence_bin = float(n_pos_bin / n) if n > 0 else np.nan

        # restrict to pseudo-positives for "recall"
        gp = g[g["ybin"] == 1]
        denom = float(gp["w"].sum())
        soft_rec = float((gp["pred"] * gp["w"]).sum() / (denom + 1e-12)) if denom > 0 else np.nan

        return pd.Series({
            "n": n,
            "n_pos_bin": n_pos_bin,
            "n_neg_bin": n_neg_bin,
            "prevalence_bin": prevalence_bin,
            "mean_pred": float(g["pred"].mean()) if n else np.nan,
            "mean_w": float(g["w"].mean()) if n else np.nan,
            "soft_recall": soft_rec,
        })

    out = (tmp.groupby(["species_id", "species_name"], as_index=False)
              .apply(agg_one, include_groups=False)
              .reset_index(drop=True))

    # nice sort
    out = out.sort_values("n", ascending=False).reset_index(drop=True)
    return out
"""
def species_soft_recall_table(df: pd.DataFrame, pred: np.ndarray, w: np.ndarray, ybin_j: np.ndarray) -> pd.DataFrame:
    """
    Per-species:
      n = total images
      n_pos_bin / n_neg_bin = binary counts from ybin_j
      prevalence_bin = n_pos_bin / n
      mean_pred = mean(pred)
      mean_w = mean(w)
      soft_recall = sum(pred*w) / sum(w) on the positive-bin subset (ybin_j==1)
    """
    out_rows = []
    for sid, g in df.assign(pred=pred, w=w, ybin=ybin_j).groupby(["species_id", "species_name"]):
        species_id, species_name = sid
        n = len(g)
        n_pos = int(g["ybin"].sum())
        n_neg = int(n - n_pos)
        prev = float(n_pos / n) if n else np.nan

        # soft recall on the "positive" subset only
        gp = g[g["ybin"] == 1]
        if len(gp) == 0:
            softrec = np.nan
        else:
            ww = gp["w"].to_numpy()
            pp = gp["pred"].to_numpy()
            softrec = float((pp * ww).sum() / (ww.sum() + 1e-12))

        out_rows.append({
            "species_id": int(species_id),
            "species_name": str(species_name),
            "n": int(n),
            "n_pos_bin": int(n_pos),
            "n_neg_bin": int(n_neg),
            "prevalence_bin": float(prev),
            "mean_pred": float(g["pred"].mean()),
            "mean_w": float(g["w"].mean()),
            "soft_recall": softrec,
        })

    return pd.DataFrame(out_rows).sort_values("n", ascending=False).reset_index(drop=True)

## Matched-pair evaluation for soft recall (new equivalent of `matched_pair_eval` + `pairs_df`)

Old matched recall gap:
- match exact counts of y==1 and y==0 between two species
- compute recall on y==1 subset

New matched soft recall gap:
- define y_bin = 1[w>0]
- match exact counts of y_bin==1 and y_bin==0 between two species
- compute soft recall on y_bin==1 subset:
    sum(pred*w)/sum(w)   (within matched positives only)

In [ ]:
def make_candidate_pairs_soft(df: pd.DataFrame, w: np.ndarray, min_each=10, max_pairs=200, seed=0):
    tmp = df[["species_id"]].copy()
    ybin = (w > 0).astype(int)
    tmp["ybin"] = ybin

    g = tmp.groupby("species_id")["ybin"].agg(["count","sum"]).rename(columns={"sum":"pos"})
    g["neg"] = g["count"] - g["pos"]
    ok = g[(g["pos"] >= min_each) & (g["neg"] >= min_each)]
    sids = ok.index.to_list()

    rng = np.random.default_rng(seed)
    pairs = []
    if len(sids) < 2:
        return pairs

    for _ in range(max_pairs * 10):
        a, b = rng.choice(sids, size=2, replace=False)
        mpos = int(min(ok.loc[a, "pos"], ok.loc[b, "pos"]))
        mneg = int(min(ok.loc[a, "neg"], ok.loc[b, "neg"]))
        if mpos >= min_each and mneg >= min_each:
            pairs.append((int(a), int(b), mpos, mneg))
        if len(pairs) >= max_pairs:
            break
    return pairs

def matched_pair_eval_soft(df: pd.DataFrame,
                           pred: np.ndarray,
                           w: np.ndarray,
                           ybin_j: np.ndarray,
                           sid_A: int, sid_B: int,
                           mpos: int, mneg: int,
                           seed=0):
    """
    Subsample each species to have EXACTLY:
      mpos positives (ybin=1) + mneg negatives (ybin=0)
    Then compute soft recall on the positive subset only:
      softrec = sum(pred*w)/sum(w) over sampled positives.
    """
    tmp = df.copy()
    tmp["pred"] = pred
    tmp["w"] = w
    tmp["ybin"] = ybin_j.astype(int)

    A = tmp[tmp.species_id == sid_A]
    B = tmp[tmp.species_id == sid_B]

    A_pos, A_neg = A[A.ybin == 1], A[A.ybin == 0]
    B_pos, B_neg = B[B.ybin == 1], B[B.ybin == 0]

    A_s = pd.concat([A_pos.sample(mpos, random_state=seed), A_neg.sample(mneg, random_state=seed)])
    B_s = pd.concat([B_pos.sample(mpos, random_state=seed), B_neg.sample(mneg, random_state=seed)])

    def softrec_on_pos(d):
        pos = d[d.ybin == 1]
        ww = pos.w.to_numpy()
        pp = pos.pred.to_numpy()
        return float((pp * ww).sum() / (ww.sum() + 1e-12)) if len(pos) else np.nan

    recA = softrec_on_pos(A_s)
    recB = softrec_on_pos(B_s)

    return {
        "sid_A": int(sid_A),
        "sid_B": int(sid_B),
        "species_A": spname(sid_A),
        "species_B": spname(sid_B),
        "npos_bin": int(mpos),
        "nneg_bin": int(mneg),
        "softrec_A": float(recA),
        "softrec_B": float(recB),
        "gap": float(abs(recA - recB)),
    }

In [ ]:
def make_ybin_from_P(P, q=0.9):
    """
    P: [N, M] raw CLIP scores (or normalized)
    Returns ybin: [N, M] in {0,1} where ybin_ij = 1 if P_ij is in top (1-q) fraction for that concept j
    """
    P = P.detach() if isinstance(P, torch.Tensor) else torch.tensor(P)
    taus = torch.quantile(P, q, dim=0)     # [M]
    ybin = (P > taus[None, :]).int()
    return ybin, taus

ybin, taus = make_ybin_from_P(P, q=0.9)

## Run one concept (new equivalent of `run_one_attribute`)

Old: trains a probe then predicts `probs`.
New: no probe. We already have:
- `pred_prob = sigmoid(test_c[:, j])`
- `w = Ptilde[:, j]`

We return:
- `info` dict (concept name, mean gap, overall soft recall, etc.)
- `pairs_df` (per matched species pair gap stats)
- `species_table` (per species soft recall table)

This mirrors the structure of your old notebook, so printed tables exist in both.

In [ ]:
def bootstrap_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    return (float(np.quantile(x, alpha/2)),
            float(np.quantile(x, 1 - alpha/2)))

def bootstrap_p_value(x):
    x = np.asarray(x, dtype=float)
    p_lo = float(np.mean(x <= 0))
    p_hi = float(np.mean(x >= 0))
    return 2.0 * min(p_lo, p_hi)

def run_one_concept_soft(
    concept_name: str,
    concept_idx: int,
    *,
    min_each: int = 10,
    n_pairs: int = 200,
    seeds=(0,1,2),
):
    pred = pred_probs[:, concept_idx]      # model score in (0,1)
    w = Ptilde[:, concept_idx]             # CLIP weight in [0,1]
    y = ybin[:, concept_idx].astype(int)   # binary labels for sampling

    # species table (overall)
    species_table = species_soft_recall_table(lf_test_df, pred, w, y)

    # overall soft recall on the positive-bin subset only
    mask = (y == 1)
    overall_soft_recall = float((pred[mask] * w[mask]).sum() / (w[mask].sum() + 1e-12)) if mask.any() else np.nan

    # matched pairs
    pairs = make_candidate_pairs_soft(lf_test_df, y, min_each=min_each, max_pairs=n_pairs, seed=0)

    rows = []
    for (a,b,mpos,mneg) in pairs:
        for sd in seeds:
            rows.append(matched_pair_eval_soft(lf_test_df, pred, w, y, a, b, mpos, mneg, seed=sd))

    res = pd.DataFrame(rows)

    if res.empty:
        pair_summary = pd.DataFrame()
        mean_gap = np.nan
        p90_gap = np.nan
    else:
        pair_summary = (
            res.groupby(["sid_A","sid_B","species_A","species_B"], as_index=False)
               .agg(
                   gap_mean=("gap","mean"),
                   gap_std=("gap","std"),
                   n_runs=("gap","size"),
                   npos_bin=("npos_bin","min"),
                   nneg_bin=("nneg_bin","min"),
                   gap_ci_lo=("gap", lambda x: bootstrap_ci(x)[0]),
                   gap_ci_hi=("gap", lambda x: bootstrap_ci(x)[1]),
                   gap_p=("gap", bootstrap_p_value),
               )
        )
        EPS = 1e-12
        pair_summary["gap_snr"] = pair_summary["gap_mean"] / (pair_summary["gap_std"].fillna(0.0) + EPS)

        mean_gap = float(res["gap"].mean())
        p90_gap = float(res["gap"].quantile(0.9))

    info = {
        "concept": concept_name,
        "j": int(concept_idx),
        "overall_soft_recall": float(overall_soft_recall),
        "n_pairs": int(len(pairs)),
        "mean_gap": float(mean_gap) if mean_gap == mean_gap else np.nan,
        "p90_gap": float(p90_gap) if p90_gap == p90_gap else np.nan,
        "mean_w": float(w.mean()),
        "pos_bin_rate": float((y == 1).mean()),
        "corr_pred_w": float(np.corrcoef(pred, w)[0,1]) if np.std(pred) > 1e-8 and np.std(w) > 1e-8 else np.nan,
    }

    return info, res, pair_summary, species_table
    
# quick test on one concept
info0, res0, summ0, st0 = run_one_concept_soft(lf_concepts_final[0], 0, n_pairs=50, seeds=(0,1))
info0, st0.head(), summ0.head()

## Screen concepts for cross-species variation (new equivalent of `screen_attributes_for_species_variation`)

Old screening produced:
- a ranked list of attributes with large cross-species recall spread.

New screening produces:
- a ranked list of *LF concepts* with large cross-species *soft recall* spread.

This is how we choose `CONCEPT_LIST`, analogous to your `ATTR_LIST`.

In [ ]:
def screen_concepts_for_species_variation(
    concepts,
    *,
    min_pos_per_species_bin: int = 10,
    min_species_with_pos: int = 15,
    min_overall_pos_bin: float = 0.05,
    max_overall_pos_bin: float = 0.95,
    max_concepts: int = 200,
    verbose_every: int = 25,
):
    rows = []
    tried = 0
    kept = 0

    for j, cname in enumerate(concepts[:max_concepts]):
        tried += 1
        pred = pred_probs[:, j]
        w = Ptilde[:, j]
        y = ybin[:, j].astype(int)

        st = species_soft_recall_table(lf_test_df, pred, w, y)

        overall_pos_bin = float((y == 1).mean())
        st_pos = st[st["n_pos_bin"] >= min_pos_per_species_bin].copy()
        n_species_pos = int(len(st_pos))

        if n_species_pos < min_species_with_pos:
            continue
        if not (min_overall_pos_bin <= overall_pos_bin <= max_overall_pos_bin):
            continue

        vals = st_pos["soft_recall"].dropna().to_numpy()
        if vals.size == 0:
            continue

        kept += 1
        rows.append({
            "concept": cname,
            "j": j,
            "overall_pos_bin": overall_pos_bin,
            "n_species_pos": n_species_pos,
            "softrec_std": float(np.std(vals)),
            "softrec_range": float(np.max(vals) - np.min(vals)),
            "softrec_p90_p10": float(np.quantile(vals, 0.9) - np.quantile(vals, 0.1)),
            "mean_w": float(w.mean()),
            "corr_pred_w": float(np.corrcoef(pred, w)[0,1]) if np.std(pred) > 1e-8 and np.std(w) > 1e-8 else np.nan,
        })

        if verbose_every and (kept % verbose_every == 0):
            print(f"[kept {kept}] {cname}  p90-p10={rows[-1]['softrec_p90_p10']:.3f}  pos_bin={overall_pos_bin:.3f}")

    screen_df = pd.DataFrame(rows)
    if screen_df.empty:
        print("No concepts passed filters. Try loosening thresholds OR check alignment.")
        print("Tried:", tried, "Kept:", kept)
        return screen_df

    screen_df = screen_df.sort_values(["softrec_p90_p10","softrec_range","softrec_std"], ascending=False).reset_index(drop=True)
    print("Tried:", tried, "Kept:", kept)
    return screen_df

lf_screen_df = screen_concepts_for_species_variation(
    lf_concepts_final,
    min_pos_per_species_bin=10,
    min_species_with_pos=15,
    min_overall_pos_bin=0.05,
    max_overall_pos_bin=0.95,
    max_concepts=250,
    verbose_every=25,
)

lf_screen_df.head(20)

## Choose CONCEPT_LIST (new equivalent of ATTR_LIST)

Old: ATTR_LIST came from screening attributes.
New: CONCEPT_LIST comes from screening LF concepts.

We keep TOP_K small initially.

In [ ]:
TOP_K = 12
CONCEPT_LIST = lf_screen_df["concept"].head(TOP_K).tolist()
CONCEPT_IDXS = lf_screen_df["j"].head(TOP_K).astype(int).tolist()

print("CONCEPT_LIST:")
for c in CONCEPT_LIST:
    print(" ", c)

## Run many concepts (new equivalent of `run_many`)

Old `run_many` printed per attribute:
- test_acc
- mean_gap

New `run_many_lf` prints per concept:
- overall_soft_recall
- mean_gap
- corr(pred, w) as a sanity check

And returns:
- `lf_info_df`
- `lf_pairs_df`
- `lf_species_df`

These correspond exactly to your old outputs, just concept-based.

In [ ]:
def run_many_lf(concept_list, concept_idxs, *, n_pairs=200, seeds=(0,1,2), min_each=10):
    all_info = []
    all_pairs = []
    all_species = []

    for cname, j in zip(concept_list, concept_idxs):
        info, res, pair_summ, species_table = run_one_concept_soft(
            cname, j,
            n_pairs=n_pairs,
            seeds=seeds,
            min_each=min_each
        )
        all_info.append(info)

        if not pair_summ.empty:
            pair_summ = pair_summ.copy()
            pair_summ["concept"] = cname
            pair_summ["j"] = j
            all_pairs.append(pair_summ)

        species_table = species_table.copy()
        species_table["concept"] = cname
        species_table["j"] = j
        all_species.append(species_table)

        print("lfcbm", cname, "softrec=", round(info["overall_soft_recall"],4),
              "mean_gap=", round(info["mean_gap"],4),
              "corr=", round(info["corr_pred_w"],4))

    lf_info_df = pd.DataFrame(all_info)
    lf_pairs_df = pd.concat(all_pairs, ignore_index=True) if all_pairs else pd.DataFrame()
    lf_species_df = pd.concat(all_species, ignore_index=True) if all_species else pd.DataFrame()
    return lf_info_df, lf_pairs_df, lf_species_df

lf_info, lf_pairs, lf_species = run_many_lf(
    CONCEPT_LIST,
    CONCEPT_IDXS,
    n_pairs=200,
    seeds=(0,1,2),
    min_each=10
)

lf_info

## Save CSVs (new equivalent of baseline_species.csv / cbm_species.csv)

Old saved:
- baseline_species.csv
- cbm_species.csv

New saves:
- lfcbm_species.csv

In [ ]:
lf_species.to_csv("lfcbm_species.csv", index=False)
lf_info.to_csv("lfcbm_info.csv", index=False)
lf_pairs.to_csv("lfcbm_pairs.csv", index=False)

lf_species.sort_values("n_pos_bin", ascending=False).head(30)

## Summarize by concept (new equivalent of `summarize_by_attr`)

Old summary table:
- gap_mean, gap_max, n_pairs, etc.

New summary table:
- same, but grouped by concept.

In [ ]:
def summarize_by_concept(pairs_df: pd.DataFrame):
    if pairs_df.empty:
        return pairs_df
    g = pairs_df.groupby(["concept"], as_index=False)
    out = g.agg(
        gap_mean=("gap_mean","mean"),
        gap_median=("gap_mean","median"),
        gap_max=("gap_mean","max"),
        n_pairs=("gap_mean","size"),
        frac_ci_above0=("gap_ci_lo", lambda s: float(np.mean(np.asarray(s) > 0))),
        frac_p_small=("gap_p", lambda s: float(np.mean(np.asarray(s) <= 0.05))),
    ).sort_values(["gap_mean"], ascending=False).reset_index(drop=True)
    return out

lf_summary = summarize_by_concept(lf_pairs)
lf_summary

## Top pairs per concept (new equivalent of `top_pairs`)

Old: `top_pairs(baseline_pairs, "baseline", attr)`
New: `top_pairs_lf(lf_pairs, concept)`

In [ ]:
def top_pairs_lf(pairs_df: pd.DataFrame, concept: str, k=10):
    sub = pairs_df[pairs_df["concept"] == concept].copy()
    if sub.empty:
        return sub

    cols = [
        "species_A","species_B",
        "gap_mean","gap_ci_lo","gap_ci_hi","gap_p",
        "gap_std","gap_snr",
        "npos_bin","nneg_bin","n_runs"
    ]
    cols = [c for c in cols if c in sub.columns]
    return sub.sort_values("gap_mean", ascending=False).head(k)[cols]

for c in CONCEPT_LIST:
    print("\nConcept:", c)
    display(top_pairs_lf(lf_pairs, c, k=10))